## Cross-Validation

Why is a single train/test split often not trustworthy?

### The problem with a single split

the result depends on:
- which rows randomly went into train
- which rows randomly went into test

A different split can give:
- different accuracy
- different precision/recall
- different conclusions

So one split can lie by chance.

### What cross-validation does

Cross-validation evaluates the model multiple times, each time on a different split.

Instead of trusting one score, we look at:
- many scores
- their average
- their spread

This gives a much more stable estimate.

### K-Fold Cross-Validation (core idea)

K-Fold means:

- split data into K equal parts (folds)
- train on K-1 folds
- test on the remaining fold
- repeat this K times

Each fold gets to be the test set once.

### Visual intuition (described)

If K = 5:

- Fold 1 → test;  others → train
- Fold 2 → test;  others → train
- Fold 3 → test;  others → train
- Fold 4 → test;  others → train
- Fold 5 → test;  others → train

You get 5 scores, not 1.

### How this relates to bias–variance

- large variation across folds → high variance
- consistently poor scores → high bias

Cross-validation exposes this clearly.

### Cross-validation in scikit-learn (basic)

#### Basic code for model training

In [43]:
import seaborn as sns

df = sns.load_dataset("tips")

X = df[["total_bill"]]
y = df["smoker"]


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)


from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [44]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5)
print(scores)

[0.63265306 0.63265306 0.6122449  0.6122449  0.5625    ]


- internally splits data into 5 folds
- trains and evaluates 5 times
- returns 5 scores

## Bias–Variance Diagnosis (with Cross-Validation)

### What you look at (two numbers)

From cross-validation, you care about:

- mean score
- variation (spread) of scores

These two together diagnose bias vs variance.

### Typical patterns and what they mean

#### Case 1: Low scores (high bias), very consistent (low variance)

eg: [50, 50.1, 49.8, 51, 49.1]
<br>
 mean = 50

- mean score is low
- scores across folds are similar

This indicates high bias.

Meaning:
- model is too simple
- cannot capture the pattern

Example causes:
- single feature only
- linear model for non-linear data

#### Case 2: High scores (low bias), very inconsistent (high variance)

eg: [60, 90, 97.6, 75, 80]
<br>
mean = 80.52

- mean score is okay or high
- scores vary a lot across folds

This indicates high variance.

Meaning:
- model is too sensitive to data
- overfitting

Example causes:
- too many parameters
- K too small in KNN
- deep decision tree

#### Case 3: High scores, consistent

eg: [89, 90, 88, 91, 90]
<br>
mean = 89.6

- mean score is high
- low variation

This is the ideal case.

Meaning:
- model generalizes well

### How to see this in code

In [45]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_train, y_train, cv=5)

print(scores.mean())    # As high and consistent as possible
print(scores.std())     # As low as possible

0.6102564102564103
0.019187986598840738


- low mean + low std → high bias
- high mean + high std → high variance
- high mean + low std → good model

### Why this matters more than accuracy

Accuracy alone cannot tell you:
- if performance will drop tomorrow
- if your model is fragile
- if you’re overfitting

Cross-validation exposes model reliability.

### How this guides next steps

Based on diagnosis:

- high bias → add features, change model
- high variance → simplify model, regularize, more data

## Hyperparameters (what they really are)

Why does changing one number suddenly fix or break the model?

### What is a hyperparameter?

A hyperparameter is:
- a value you set before training
- it controls how the model learns
- it is not learned from data

Examples already used:
- K in KNN
- max_depth in Decision Tree
- C in Logistic Regression

### Hyperparameters vs model parameters

**Model parameters:**
- learned from data

examples:
- slope in Linear Regression
- weights in Logistic Regression

**Hyperparameters:**
- chosen by you
- control bias–variance

examples:
- n_neighbors
- max_depth

Think of it as:

parameters are learned, hyperparameters are decisions

### How hyperparameters affect bias–variance (core idea)

Example: KNN

 small K
- very flexible
- low bias
- high variance

 large K
- very smooth
- high bias
- low variance

### So how do we choose hyperparameters?

Not by guessing.

We:
- try many values
- evaluate them using cross-validation
- choose the one with the best average performance

This is called hyperparameter tuning.

### Manual tuning (conceptual)

Example:
- try K = 1, 3, 5, 7, 9
- compare cross-validation scores
- pick the most stable + best mean

We already did this earlier with KNN.

## GridSearchCV (Automatic Hyperparameter Tuning)

“Which hyperparameter combination works best on average, not just once?”

We’ll use KNN, because:

- We already understand K

- tuning effect is obvious

### Break the name into parts

#### Grid

- you define a grid of values
- example:
- K = [3, 5, 7, 9]
- weights = [uniform, distance]

Grid = all possible combinations.

#### Search

- the model is trained multiple times
- once for each combination
- nothing is guessed

#### CV (Cross-Validation)

- each combination is evaluated using K-fold cross-validation
- you don’t trust a single split
- you trust the average performance

### What GridSearchCV actually does

For every hyperparameter combination:

- split training data into folds
- train on K-1 folds
- validate on the remaining fold
- repeat for all folds
- compute the mean score

Then:
- compare all combinations
- select the best average performer

### Code

#### Step 1: Data (same tips dataset)

In [46]:
import seaborn as sns

df = sns.load_dataset("tips")

X = df[["total_bill", "size"]]
y = df["smoker"].map({"Yes":1, "No":0})

#### Step 2: Train/test split

In [47]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

#### Step 3: Pipeline (so scaling + model are safe)

In [48]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

- "scaler" is just a label (you choose it)
- StandardScaler() is the actual transformer

This name is important later for GridSearch.

Why pipeline?

- scaling is required for KNN
- GridSearch will refit safely (no leakage)

StandardScaler()
- computes mean and standard deviation from training data
- scales numeric features
- outputs scaled data

KNeighborsClassifier()
- takes the scaled data
- trains the KNN model
- later uses neighbors to predict

when you call .fit()

Internally, this happens automatically:

- StandardScaler.fit(X_train)
- StandardScaler.transform(X_train)
- KNN.fit(scaled_X_train, y_train)

for every fold

#### Step 4: Define hyperparameter grid

In [49]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"]
}

Do all neighbors vote equally, or do closer neighbors matter more?

That’s what weights decides.

**weights="uniform" (default behavior)**

- Every one of the K neighbors gets the same vote

**weights="distance" (smarter behavior)**

- Closer neighbors get more influence than farther ones

$$
\text{weight} = \frac{1}{\text{distance}}
$$


- very small distance → large weight

- large distance → small weight


Simple example (this is the key intuition)

Assume K = 3.

Distances to neighbors:

- Neighbor A (class 1): distance = 0.1

- Neighbor B (class 0): distance = 0.2

- Neighbor C (class 0): distance = 2.0

With distance weights:

- class 1 vote ≈ very strong

- class 0 votes ≈ weak (especially the far one)

- prediction → class 1


- knn__ refers to the step name in the pipeline
- we are tuning model behavior, not code

#### Step 5: Run GridSearchCV

In [50]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv = 5,
    scoring = "f1",
    n_jobs = -1
)

grid.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'knn__n_neighbors': [3, 5, ...], 'knn__weights': ['uniform', 'distance']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


**estimator=pipe**

“This is the thing you should train repeatedly.”

**param_grid=param_grid**

“These are the values you are allowed to try.”

This means GridSearch will try ALL combinations:

- K=3, weights=uniform
- K=3, weights=distance
- K=5, weights=uniform
- K=5, weights=distance
- … and so on

**cv=5**

“Evaluate each combination using 5-fold cross-validation.”

So for every single combination:

- split training data into 5 folds
- train 5 times
- validate 5 times
- compute the mean score

**scoring="f1"**

When GridSearch compares combinations, it asks:

“Which one has the highest average F1 score across folds?”

**n_jobs=-1**

This controls parallelism.

Meaning:
- how many CPU cores GridSearch is allowed to use

Values:
- n_jobs=1 → use 1 core (slow)
- n_jobs=2 → use 2 cores
- n_jobs=-1 → use all available cores

#### Step 6: Inspect results (THIS IS THE PAYOFF)

In [51]:
grid.best_params_

{'knn__n_neighbors': 5, 'knn__weights': 'uniform'}

In [52]:
grid.best_score_    # f-1 score

np.float64(0.4193295271556141)

#### Step 7: Final evaluation on test data

In [53]:
from sklearn.metrics import classification_report

y_pred = grid.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.71      0.87      0.78        31
           1       0.64      0.39      0.48        18

    accuracy                           0.69        49
   macro avg       0.67      0.63      0.63        49
weighted avg       0.68      0.69      0.67        49



## RandomSearchCV

RandomSearch tries random combinations from a range. It trades completeness for speed.

In practice:

- much faster
- almost same performance
- preferred for big models

### Why RandomSearchCV exists

GridSearch time grows like:

$$
\text{combinations} \times \text{cv} \times \text{fit time}
$$

This becomes infeasible very quickly.

**RandomSearchCV:**
- limits how many combinations are tried
- explores a wider space
- finishes much faster
- usually finds near-optimal parameters

### When RandomSearchCV is better than GridSearch

- large parameter ranges
- continuous parameters
- long-running models
- real-world datasets

**GridSearch** is mainly for:
- teaching
- very small grids

### Parameter space

GridSearch uses fixed lists. <br>
RandomSearch uses distributions.

#### Step 1: Define parameter distributions

In [57]:
from scipy.stats import randint

param_dist = {
    "knn__n_neighbors": randint(3, 30),
    "knn__weights": ["uniform", "distance"]
}

- randint(3, 30) means random integer between 3 and 29
- values are sampled, not enumerated
- this creates a much larger search space

#### Step 2: Create RandomSearchCV

In [58]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator = pipe,
    param_distributions = param_dist,
    n_iter = 20,
    cv = 5,
    scoring = "f1",
    n_jobs = -1,
    random_state = 42
)

estimator=pipe
- the full pipeline (scaler + model)

param_distributions=param_dist
- where parameters are sampled from

n_iter=20
- number of random combinations tried

cv=5
- 5-fold cross-validation

scoring="f1"
- optimize F1 for positive class

n_jobs=-1
- use all available logical cores

random_state=42
- reproducible randomness

#### Step 3: Fit RandomSearchCV

In [59]:
random_search.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_distributions,"{'knn__n_neighbors': <scipy.stats....001BC53B12090>, 'knn__weights': ['uniform', 'distance']}"
,n_iter,20
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


- 20 random parameter combinations
- each evaluated with 5-fold CV
- total fits = 20 × 5 = 100

#### Step 4: Best parameters and score

In [60]:
print(random_search.best_params_)
print(random_search.best_score_)

{'knn__n_neighbors': 14, 'knn__weights': 'uniform'}
0.3730718099596818


#### Step 5: Final evaluation on test set

In [62]:
from sklearn.metrics import classification_report

y_pred = random_search.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.63      0.94      0.75        31
           1       0.33      0.06      0.10        18

    accuracy                           0.61        49
   macro avg       0.48      0.50      0.42        49
weighted avg       0.52      0.61      0.51        49



RandomSearchCV gives most of GridSearch’s benefit at a fraction of the cost.